#### Cel: wyznaczenie częstotliwości drgań własnych maszyny podczas wybiegu

#### Z uwagi na brak pomiaru z tachometru, klasyczne śledzenie rzędów stało się niemożliwe, zastosowano ekstrakcję profilu prędkości obrotowej bezpośrednio z sygnału drganiowego

#### Określamy kinematykę układu - używamy STFT do wygenerowania spektrogramu 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, medfilt, find_peaks


def analyze_rundown_tacholess(file_path, title, fs=32768, min_hz=10, max_hz=140):
    print("Wczytywanie danych...")
    df = pd.read_csv(file_path)
    
    time = df['Time_s'].values
    acc_2 = df['ACXXX_SN2 [g]'].values
    
    # 1. Obliczenie Spektrogramu
    nperseg = 8192
    noverlap = int(nperseg * 0.75) # Zwiększamy nałożenie (overlap), aby otrzymać gładszy wykres
    f_stft, t_stft, Sxx = spectrogram(acc_2, fs=fs, window='hann', 
                                      nperseg=nperseg, noverlap=noverlap, scaling='spectrum')
    
    # 2. Ekstrakcja Wirtualnego Tachometru z zawężonym oknem
    virtual_tacho_hz = []
    virtual_tacho_amp = []
    
    # Tworzymy przedział ufności: szukamy tylko między min_hz a max_hz
    search_mask = (f_stft >= min_hz) & (f_stft <= max_hz)
    f_search = f_stft[search_mask]
    Sxx_search = Sxx[search_mask, :]
    
    for i in range(Sxx_search.shape[1]):
        idx_max = np.argmax(Sxx_search[:, i])
        virtual_tacho_hz.append(f_search[idx_max])
        virtual_tacho_amp.append(Sxx_search[idx_max, i])
        
    virtual_tacho_hz = np.array(virtual_tacho_hz)
    virtual_tacho_amp = np.array(virtual_tacho_amp)
    
    # Filtracja medianowa, aby usunąć ewentualne "skoki" i wygładzić schodki
    virtual_tacho_hz = medfilt(virtual_tacho_hz, kernel_size=11)
    
    # 3. Znalezienie rezonansu 
    idx_resonance = np.argmax(virtual_tacho_amp)
    res_time = t_stft[idx_resonance]
    res_freq = virtual_tacho_hz[idx_resonance]
    
    # 4. Wykreślenie
    plt.figure(figsize=(12, 6))
    f_max_plot = max_hz + 20
    freq_mask = f_stft <= f_max_plot
    
    plt.pcolormesh(t_stft, f_stft[freq_mask], 10 * np.log10(Sxx[freq_mask, :]), shading='gouraud', cmap='viridis')
    plt.colorbar(label='Amplituda [dB]')
    
    plt.plot(t_stft, virtual_tacho_hz, color='red', linewidth=2, label='Wirtualne Tacho')
    plt.scatter(res_time, res_freq, color='white', edgecolors='black', s=100, zorder=5, label=f'Max Amplituda ({res_freq:.1f} Hz)')
    
    plt.title(f'Spektrogram wybiegu 1X, {title}')
    plt.ylabel('Częstotliwość [Hz]')
    plt.xlabel('Czas [s]')
    plt.ylim(0, f_max_plot)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Uruchomienie z nowymi zakresami:
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a5_zew_processed.csv', title='a5_zew', min_hz=10, max_hz=140)
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_sr_processed.csv', title='a7_sr', min_hz=10, max_hz=140)
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew_processed.csv', title='a7_zew', min_hz=10, max_hz=140)
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew2_processed.csv',title='a7_zew2',  min_hz=10, max_hz=140)
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_sr_processed.csv', title='a11_sr',  min_hz=10, max_hz=140)
analyze_rundown_tacholess(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_zew_sr_processed.csv',title='a11_zew_sr', min_hz=10, max_hz=140)


#### Jak widać z powyższych wykresów globalne maksimum aplitudy zlokalizowane jest w początkowej fazie pomiaru. Nie jest to jednak rezonans, lecz odpowiedź sztywna układu na dużą siłę odśrodkową. 

#### Z tego względu aby zidentyfikować rezonans, generujemy krzywą amplitudową pierwszej harominczej, następnie poszukujemy wyraźnego lokalnego wzrostu amplitudy drgań w paśmie niższych prędkości przy opadającej sile wymuszającej, co stanowi o przejściu przez strefę drgań własnych

In [ ]:
def analyze_and_plot_rundown(file_path, title, fs=32768, min_hz=10, max_hz=140, search_max_hz=50):
    print(f"Wczytywanie danych z: {file_path} ...")
    df = pd.read_csv(file_path)
    acc_2 = df['ACXXX_SN2 [g]'].values
    
    # 1. Obliczenie Spektrogramu
    nperseg = 8192
    noverlap = int(nperseg * 0.75)
    f_stft, t_stft, Sxx = spectrogram(acc_2, fs=fs, window='hann', 
                                      nperseg=nperseg, noverlap=noverlap, scaling='spectrum')
    
    virtual_tacho_hz = []
    virtual_tacho_amp = []
    
    search_mask = (f_stft >= min_hz) & (f_stft <= max_hz)
    f_search = f_stft[search_mask]
    Sxx_search = Sxx[search_mask, :]
    
    # Ekstrakcja 1X
    for i in range(Sxx_search.shape[1]):
        idx_max = np.argmax(Sxx_search[:, i])
        virtual_tacho_hz.append(f_search[idx_max])
        virtual_tacho_amp.append(np.sqrt(Sxx_search[idx_max, i]))
        
    virtual_tacho_hz = medfilt(virtual_tacho_hz, kernel_size=11)
    virtual_tacho_amp = np.array(virtual_tacho_amp)
    
    # 2. Wygladzanie 
    smoothed_amp_time = medfilt(virtual_tacho_amp, kernel_size=21)
    
    # Poszukiwanie rezonansu
    lower_band_mask = virtual_tacho_hz <= search_max_hz
    hz_lower = virtual_tacho_hz[lower_band_mask]
    amp_lower = smoothed_amp_time[lower_band_mask]
    t_lower = t_stft[lower_band_mask]
    
    peaks, _ = find_peaks(amp_lower, prominence=0.005, distance=10)
    
    res_hz = None
    if len(peaks) > 0:
        best_peak_idx = peaks[np.argmax(amp_lower[peaks])]
        res_hz = hz_lower[best_peak_idx]
        res_amp = amp_lower[best_peak_idx]
        res_time = t_lower[best_peak_idx]
        
        window_start = max(0, res_time - 1.5)
        window_end = res_time + 1.5
        
        print("-" * 50)
        print("ZIDENTYFIKOWANO REZONANS LOKALNY:")
        print(f"Częstotliwość: ~{res_hz:.2f} Hz")
        print(f"Czas wystąpienia: ~{res_time:.2f} s")
        print(f"GOTOWE PARAMETRY DO FFT: start_time = {window_start:.2f}, end_time = {window_end:.2f}")
        print("-" * 50)
    else:
        print("-" * 50)
        print("UWAGA: Nie znaleziono wyraźnego rezonansu w dolnym paśmie.")
        print("-" * 50)

    # 3. Sortowanie do wykresu krzywej pierwszej harmonicznej
    sort_idx = np.argsort(virtual_tacho_hz)
    sorted_hz = virtual_tacho_hz[sort_idx]
    sorted_amp = smoothed_amp_time[sort_idx]
    
    # 4. Rysowanie wykresu
    plt.figure(figsize=(11, 6))
    plt.plot(sorted_hz, sorted_amp, color='blue', linewidth=2, label='Amplituda 1X podczas wybiegu')
    
    # Zaznaczenie rezonansu (lokalnego maksimum)
    if res_hz is not None:
        plt.scatter(res_hz, res_amp, color='red', s=120, zorder=5, 
                    label=f'Rezonans układu: {res_hz:.1f} Hz (Czas: {res_time:.1f}s)')
        plt.axvline(x=res_hz, color='red', linestyle='--', alpha=0.5)
    
    # Zaznaczenie globalnego maksimum siły odśrodkowej
    max_idx = np.argmax(sorted_amp)
    max_hz_global = sorted_hz[max_idx]
    max_amp_global = sorted_amp[max_idx]
    
    if res_hz is None or max_hz_global != res_hz:
        plt.scatter(max_hz_global, max_amp_global, color='gray', s=60, zorder=4, 
                    label=f'Maksymalna siła odśrodkowa: {max_hz_global:.1f} Hz')
    
    plt.title(f'Śledzenie 1. Harmonicznej (1X) - Detekcja Prędkości Krytycznej, {title}')
    plt.xlabel('Prędkość obrotowa (1X) [Hz]')
    plt.ylabel('Amplituda drgań [g] (RMS)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.xlim(min_hz, max_hz)
    plt.tight_layout()
    plt.show()

In [ ]:
# Uruchomienie z nowymi zakresami:
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a5_zew_processed.csv', title='a5_zew', min_hz=10, max_hz=140)
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_sr_processed.csv', title='a7_sr',min_hz=10, max_hz=140)
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew_processed.csv', title='a7_zew', min_hz=10, max_hz=140)
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew2_processed.csv', title='a7_zew2', min_hz=10, max_hz=140)
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_sr_processed.csv',  title='a11_sr',min_hz=10, max_hz=140)
analyze_and_plot_rundown(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_zew_sr_processed.csv', title='a11_zew_sr', min_hz=10, max_hz=140)

In [ ]:

def compute_windowed_fft(df, start_time, end_time, title, fs=32768):
    """
    Funkcja do obliczenia FFT dla wybranego okna czasowego (np. na rezonansie).
    """
    # Wycięcie interesującego nas fragmentu
    mask = (df['Time_s'] >= start_time) & (df['Time_s'] <= end_time)
    window_data = df.loc[mask, 'ACXXX_SN1 [g]'].values
    
    # Usunięcie składowej stałej 
    window_data = window_data - np.mean(window_data)
    
    if len(window_data) == 0:
        print("Błąd: Puste okno czasowe.")
        return
        
    # Nałożenie okna Hanna (aby zminimalizować wyciek widma)
    window = np.hanning(len(window_data))
    windowed_signal = window_data * window
    
    # Obliczenie FFT
    N = len(windowed_signal)
    yf = np.fft.rfft(windowed_signal)
    xf = np.fft.rfftfreq(N, 1 / fs)
    
    # Normalizacja amplitudy
    amplitude = 2.0 / N * np.abs(yf)
    
    plt.figure(figsize=(10, 5))
    plt.plot(xf, amplitude)
    plt.title(f'Widmo amplitudowe (FFT) dla okna: {start_time}s - {end_time}s, {title}')
    plt.xlabel('Częstotliwość [Hz]')
    plt.ylabel('Amplituda [g]')
    plt.xlim(0, 500) 
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
df_a5_zew = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a5_zew_processed.csv')
df_a7_sr = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_sr_processed.csv')
df_a7_zew = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew_processed.csv')
df_a7_zew2 = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew2_processed.csv')
df_a11_sr = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_sr_processed.csv')
df_a11_zew_sr = pd.read_csv(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_zew_sr_processed.csv')
compute_windowed_fft(df_a5_zew, start_time=52.38, end_time=55.38, title="a5_zew")
compute_windowed_fft(df_a7_sr, start_time=50.38, end_time=53.38, title="a7_sr")
compute_windowed_fft(df_a7_zew, start_time=46.31, end_time=49.31, title="a7_zew")
compute_windowed_fft(df_a7_zew2, start_time=45.44, end_time=48.44, title="a7_zew2")
# Tutaj nie wykryto min lokalnego
#compute_windowed_fft(df_a11_sr, start_time=48.19, end_time=51.19, title="a11_sr")
compute_windowed_fft(df_a11_zew_sr, start_time=51.50, end_time=54.50, title="a11_zew_sr")

---
#### Analiza z wykorzystaniem sygnału z tachometru
#### W próbkach **a5_zew** i **a11_zew_sr** dostępny jest sygnał impulsowy z tachometru (kolumna Tacho_Ch8 [Hz]). Kolumna zawiera niezerową wartość przy każdym obrocie wału — rzeczywistą prędkość obrotową wyznaczamy z odstępów czasowych między kolejnymi impulsami (1/Δt).

In [ ]:
from scipy.interpolate import interp1d


def extract_tacho_speed_profile(df, fs=32768):
    """
    Wyznacza profil prędkości obrotowej z impulsów tachometru.
    Zwraca (t_freq, freq_hz) — czas i odpowiadająca mu częstotliwość 1X [Hz].
    Częstotliwość obliczana jest jako odwrotność odstępu między sąsiednimi impulsami.
    """
    tacho = df['Tacho_Ch8 [Hz]'].values
    time = df['Time_s'].values

    pulse_times = time[tacho > 0]

    if len(pulse_times) < 2:
        return None, None

    intervals = np.diff(pulse_times)
    freq_hz = 1.0 / intervals
    t_freq = 0.5 * (pulse_times[:-1] + pulse_times[1:])

    return t_freq, freq_hz


def analyze_rundown_with_tacho(file_path, title, fs=32768, min_hz=10, max_hz=140, search_max_hz=50):
    """
    Detekcja rezonansu podczas wybiegu z wykorzystaniem sygnału tachometru.
    Analogiczna do analyze_and_plot_rundown, ale profil prędkości pochodzi
    z rzeczywistych impulsów tachometru zamiast wirtualnego tacho z STFT.
    """
    print(f"Wczytywanie danych z: {file_path} ...")
    df = pd.read_csv(file_path)
    acc_2 = df['ACXXX_SN2 [g]'].values

    # Ekstrakcja profilu prędkości z tachometru
    t_freq, freq_hz = extract_tacho_speed_profile(df, fs=fs)
    if t_freq is None:
        print("UWAGA: Brak sygnału tachometru w tym pliku.")
        return

    freq_hz_smooth = medfilt(freq_hz, kernel_size=11)
    print(f"Zakresy tachometru: {freq_hz_smooth.min():.1f} – {freq_hz_smooth.max():.1f} Hz, impulsów: {len(freq_hz)+1}")

    # Spektrogram
    nperseg = 8192
    noverlap = int(nperseg * 0.75)
    f_stft, t_stft, Sxx = spectrogram(acc_2, fs=fs, window='hann',
                                       nperseg=nperseg, noverlap=noverlap, scaling='spectrum')

    # Interpolacja profilu tachometru na siatkę czasową STFT
    tacho_interp_fn = interp1d(t_freq, freq_hz_smooth, bounds_error=False, fill_value=(freq_hz_smooth[0], freq_hz_smooth[-1]))
    tacho_at_stft = tacho_interp_fn(t_stft)
    tacho_at_stft = np.clip(tacho_at_stft, min_hz, max_hz)

    # Ekstrakcja amplitudy 1X z STFT wzdłuż profilu tachometru
    amp_1x = np.array([
        np.sqrt(Sxx[np.argmin(np.abs(f_stft - f_rot)), i])
        for i, f_rot in enumerate(tacho_at_stft)
    ])
    amp_1x_smooth = medfilt(amp_1x, kernel_size=21)

    # Poszukiwanie rezonansu w dolnym paśmie prędkości
    lower_band_mask = tacho_at_stft <= search_max_hz
    hz_lower = tacho_at_stft[lower_band_mask]
    amp_lower = amp_1x_smooth[lower_band_mask]
    t_lower = t_stft[lower_band_mask]

    peaks, _ = find_peaks(amp_lower, prominence=0.005, distance=10)

    res_hz = None
    if len(peaks) > 0:
        best_peak_idx = peaks[np.argmax(amp_lower[peaks])]
        res_hz = hz_lower[best_peak_idx]
        res_amp = amp_lower[best_peak_idx]
        res_time = t_lower[best_peak_idx]

        window_start = max(0, res_time - 1.5)
        window_end = res_time + 1.5

        print("-" * 50)
        print("ZIDENTYFIKOWANO REZONANS LOKALNY (TACHO):")
        print(f"Częstotliwość: ~{res_hz:.2f} Hz")
        print(f"Czas wystąpienia: ~{res_time:.2f} s")
        print(f"GOTOWE PARAMETRY DO FFT: start_time = {window_start:.2f}, end_time = {window_end:.2f}")
        print("-" * 50)
    else:
        print("-" * 50)
        print("UWAGA: Nie znaleziono wyraźnego rezonansu w dolnym paśmie.")
        print("-" * 50)

    # Rysowanie
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    # Wykres 1: Spektrogram z nałożoną krzywą tachometru
    f_max_plot = max_hz + 20
    freq_mask = f_stft <= f_max_plot
    axes[0].pcolormesh(t_stft, f_stft[freq_mask], 10 * np.log10(Sxx[freq_mask, :]),
                       shading='gouraud', cmap='viridis')
    axes[0].plot(t_stft, tacho_at_stft, color='red', linewidth=2, label='Tacho 1X (rzeczywiste)')
    if res_hz is not None:
        axes[0].scatter(res_time, res_hz, color='white', edgecolors='black', s=100, zorder=5,
                        label=f'Rezonans ({res_hz:.1f} Hz)')
    axes[0].set_title(f'Spektrogram wybiegu 1X z Tachometrem, {title}')
    axes[0].set_ylabel('Częstotliwość [Hz]')
    axes[0].set_xlabel('Czas [s]')
    axes[0].set_ylim(0, f_max_plot)
    axes[0].legend()

    # Wykres 2: Krzywa amplitudowa 1X vs prędkość obrotowa
    sort_idx = np.argsort(tacho_at_stft)
    sorted_hz = tacho_at_stft[sort_idx]
    sorted_amp = amp_1x_smooth[sort_idx]

    axes[1].plot(sorted_hz, sorted_amp, color='blue', linewidth=2, label='Amplituda 1X podczas wybiegu')
    if res_hz is not None:
        axes[1].scatter(res_hz, res_amp, color='red', s=120, zorder=5,
                        label=f'Rezonans układu: {res_hz:.1f} Hz (Czas: {res_time:.1f}s)')
        axes[1].axvline(x=res_hz, color='red', linestyle='--', alpha=0.5)

    max_idx = np.argmax(sorted_amp)
    if res_hz is None or abs(sorted_hz[max_idx] - res_hz) > 1.0:
        axes[1].scatter(sorted_hz[max_idx], sorted_amp[max_idx], color='gray', s=60, zorder=4,
                        label=f'Maksymalna siła odśrodkowa: {sorted_hz[max_idx]:.1f} Hz')

    axes[1].set_title(f'Śledzenie 1. Harmonicznej (1X, Tacho) - Detekcja Prędkości Krytycznej, {title}')
    axes[1].set_xlabel('Prędkość obrotowa (1X) [Hz]')
    axes[1].set_ylabel('Amplituda drgań [g] (RMS)')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()
    axes[1].set_xlim(min_hz, max_hz)

    plt.tight_layout()
    plt.show()

In [ ]:
# Uruchomienie analizy z tachometrem 
analyze_rundown_with_tacho(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a5_zew_processed.csv', title='a5_zew', min_hz=10, max_hz=140)
analyze_rundown_with_tacho(r'E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_zew_sr_processed.csv', title='a11_zew_sr', min_hz=10, max_hz=140)

---
## Klasyfikacja niewyważenia — inferencja modeli z pipeline na danych wybiegowych

Pipeline (Przygotowanie_Danych_V3 → Normalizacja_Danych_V3 → Modele_V3):
- **non_tacho_model** (6 kanałów: SN1 + SN2 × {amplitude_db, phase_sin, phase_cos}) → wartość niewyważenia **{0, 1, 2, 3}** (0=wyważone, 1=zew, 2=zew2, 3=zew_sr)
- **tacho_model** (9 kanałów: SN1 + SN2 + Tacho × {amplitude_db, phase_sin, phase_cos}) → kąt niewyważenia jako [sin, cos]


**Uwaga interpretacyjna**: dane treningowe to pomiary przy stałej prędkości. Wybiegi to pomiary niestacjonarne (prędkość opada). Aby zbliżyć warunki do treningowych, domyślnie używamy pierwszych **WINDOW_S** sekund każdego wybiegu (faza quasi-ustalona, zanim nastąpi wyraźne hamowanie).

In [ ]:
import json
import torch
from torch import nn
from pathlib import Path


PIPELINE_MODELS_DIR = Path(r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\models\Wywazanie\Pipeline")

# Okno czasowe pobierane z początku wybiegu [s] – faza quasi-ustalona (maszyną jeszcze nie zwalnia dynamicznie)
WINDOW_S = 2.0

# Parametry STFT jak w Przygotowanie_Danych_V3
PIPELINE_NPERSEG = 2 ** 15          # 32768 próbek
PIPELINE_NOVERLAP = PIPELINE_NPERSEG // 2
PIPELINE_F_MIN_HZ = 1
PIPELINE_F_MAX_HZ = 5000

# Mapowanie lokalizacji na wartość niewyważenia (zgodnie z Normalizacja_Danych_V3)
LOCATION_TO_IMBALANCE = {
    "baseline": 0,
    "sr":   0,
    "zew":  1,
    "zew2": 2,
    "zew_sr": 3,
}

# Pliki wybiegowe z etykietami wyciągniętymi z nazw plików
WYBIEG_FILES = [
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a5_zew_processed.csv",
        "title": "a5_zew",
        "angle": 5,
        "location": "zew",
        "true_imbalance": LOCATION_TO_IMBALANCE["zew"],   # = 1
        "has_tacho": True,
    },
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_sr_processed.csv",
        "title": "a7_sr",
        "angle": 7,
        "location": "sr",
        "true_imbalance": LOCATION_TO_IMBALANCE["sr"],    # = 0
        "has_tacho": False,
    },
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew_processed.csv",
        "title": "a7_zew",
        "angle": 7,
        "location": "zew",
        "true_imbalance": LOCATION_TO_IMBALANCE["zew"],   # = 1
        "has_tacho": False,
    },
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a7_zew2_processed.csv",
        "title": "a7_zew2",
        "angle": 7,
        "location": "zew2",
        "true_imbalance": LOCATION_TO_IMBALANCE["zew2"],  # = 2
        "has_tacho": False,
    },
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_sr_processed.csv",
        "title": "a11_sr",
        "angle": 11,
        "location": "sr",
        "true_imbalance": LOCATION_TO_IMBALANCE["sr"],    # = 0
        "has_tacho": False,
    },
    {
        "path": r"E:\NewWorkspaceVSC\3_Term\Studies_AI_3_Term\Zespolowy\wywazanie_napedu\data\processed\wybieg\wybieg_a11_zew_sr_processed.csv",
        "title": "a11_zew_sr",
        "angle": 11,
        "location": "zew_sr",
        "true_imbalance": LOCATION_TO_IMBALANCE["zew_sr"], # = 3
        "has_tacho": True,
    },
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Urządzenie: {DEVICE}")
print(f"Katalog modeli: {PIPELINE_MODELS_DIR}")

In [ ]:
def build_spectrogram_tensor(df, fs, channels, f_min_hz=PIPELINE_F_MIN_HZ, f_max_hz=PIPELINE_F_MAX_HZ,
                             window_s=None):
    """
    Przelicza kolumny sygnału z DataFrame na tensor X używany przez modele CNN.

    Dla każdego kanału oblicza STFT (z tymi samymi parametrami co pipeline),
    a następnie wyciąga: amplitude_db, phase_sin, phase_cos.
    Opcjonalne `window_s` skraca sygnał do pierwszych N sekund — przybliża warunki
    pomiarów ustalonych, na których modele były trenowane.

    Zwraca:
        X             – ndarray [C, F, T], float32
        component_names – lista nazw kanałów w kolejności odpowiadającej osi C
        frequency_hz  – ndarray [F], osie częstotliwości (Hz)
    """
    arrays = []
    component_names = []

    frequency_hz = None

    for col in channels:
        signal = df[col].values.astype(np.float64)

        # Opcjonalne przycięcie do okna quasi-ustalonego
        if window_s is not None:
            n_samples = int(window_s * fs)
            signal = signal[:n_samples]

        # STFT – tryb complex zgodny z pipeline
        f, t, Zxx = spectrogram(
            signal,
            fs=fs,
            window="hann",
            nperseg=PIPELINE_NPERSEG,
            noverlap=PIPELINE_NOVERLAP,
            detrend="constant",
            scaling="spectrum",
            mode="complex",
        )

        # Maska częstotliwości (1–5000 Hz), zgodna z F_MIN/F_MAX pipeline
        freq_mask = (f >= f_min_hz) & (f <= f_max_hz)
        f_plot = f[freq_mask]
        Z = Zxx[freq_mask, :]

        if frequency_hz is None:
            frequency_hz = f_plot.astype(np.float32)

        # Trzy reprezentacje, zgodnie z Przygotowanie_Danych_V3
        amp_db = 20.0 * np.log10(np.abs(Z) + np.finfo(float).eps)
        phase = np.angle(Z)

        arrays.append(amp_db.astype(np.float32))
        arrays.append(np.sin(phase).astype(np.float32))
        arrays.append(np.cos(phase).astype(np.float32))

        component_names += [
            f"{col} | amplitude_db",
            f"{col} | phase_sin",
            f"{col} | phase_cos",
        ]

    X = np.stack(arrays, axis=0)   # [C, F, T]
    return X, component_names, frequency_hz


def apply_pipeline_normalization(X, component_names, norm_stats):
    """
    Standaryzuje kanały amplitude_db z wykorzystaniem statystyk z pliku normalization_info.json.

    Tylko kanały SN1 i SN2 amplitude_db są normalizowane; phase_sin, phase_cos i tacho
    pozostają niezmienione — zgodnie z polityką Normalizacja_Danych_V3.
    """
    X = X.copy().astype(np.float32)

    norm_names = norm_stats["normalized_component_names"]
    means = np.array(norm_stats["mean"], dtype=np.float32)
    stds  = np.array(norm_stats["std"],  dtype=np.float32)

    for local_idx, comp_name in enumerate(norm_names):
        if comp_name in component_names:
            channel_idx = component_names.index(comp_name)
            X[channel_idx] = (X[channel_idx] - means[local_idx]) / stds[local_idx]

    return X

In [ ]:
class SpectrogramCNNRegressor(nn.Module):
    """
    Skopiowana definicja modelu z Modele_V3.ipynb (architektura 'large').
    Konieczna do załadowania zapisanych wag przez torch.load().
    AdaptiveAvgPool2d(1,1) sprawia, że model akceptuje dowolny rozmiar F×T.
    """

    def __init__(self, in_channels, output_dim, dropout=0.35):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.regressor(self.pool(self.features(x)))


def _strip_model_prefix(state_dict):
    """
    Modele były zapisane jako TachoAngleSinCosCNN/NonTachoImbalanceCNN,
    które przechowują SpectrogramCNNRegressor jako self.model.
    Stąd klucze w state_dict mają prefix 'model.' — obcinamy go,
    aby załadować wagi bezpośrednio do SpectrogramCNNRegressor.
    """
    return {k.replace("model.", "", 1): v for k, v in state_dict.items()}


def load_pipeline_models(pipeline_dir):
    """
    Wczytuje wagi modeli i statystyki normalizacji z katalogu pipeline.

    Zwraca słownik z kluczami:
      'non_tacho_model'   – model CNN do predykcji niewyważenia
      'tacho_model'       – model CNN do predykcji kąta (sin/cos)
      'norm_non_tacho'    – statystyki normalizacji dla danych bez tacho
      'norm_tacho'        – statystyki normalizacji dla danych z tacho
    """
    pipeline_dir = Path(pipeline_dir)

    # Wczytanie statystyk normalizacji z JSON
    norm_info_path = pipeline_dir / "Dataset_Normalzied" / "normalization_info.json"
    with open(norm_info_path, "r", encoding="utf-8") as f:
        norm_info = json.load(f)

    norm_non_tacho = norm_info["non_tacho"]
    norm_tacho     = norm_info["tacho"]

    # Wczytanie modelu niewyważenia (6 kanałów → 1 wartość)
    nt_ckpt = torch.load(
        pipeline_dir / "Torch_Models" / "non_tacho_imbalance_cnn_regressor.pt",
        map_location=DEVICE,
        weights_only=False,
    )
    non_tacho_model = SpectrogramCNNRegressor(
        in_channels=nt_ckpt["in_channels"],
        output_dim=nt_ckpt["output_dim"],
    ).to(DEVICE)
    non_tacho_model.load_state_dict(_strip_model_prefix(nt_ckpt["model_state_dict"]))
    non_tacho_model.eval()

    # Wczytanie modelu kąta (9 kanałów → 2 wartości: sin, cos)
    ta_ckpt = torch.load(
        pipeline_dir / "Torch_Models" / "tacho_angle_sincos_cnn_regressor.pt",
        map_location=DEVICE,
        weights_only=False,
    )
    tacho_model = SpectrogramCNNRegressor(
        in_channels=ta_ckpt["in_channels"],
        output_dim=ta_ckpt["output_dim"],
    ).to(DEVICE)
    tacho_model.load_state_dict(_strip_model_prefix(ta_ckpt["model_state_dict"]))
    tacho_model.eval()

    print(f"Wczytano modele z: {pipeline_dir}")
    print(f"  non_tacho: in_channels={nt_ckpt['in_channels']}, output_dim={nt_ckpt['output_dim']}")
    print(f"  tacho:     in_channels={ta_ckpt['in_channels']}, output_dim={ta_ckpt['output_dim']}")

    return {
        "non_tacho_model": non_tacho_model,
        "tacho_model": tacho_model,
        "norm_non_tacho": norm_non_tacho,
        "norm_tacho": norm_tacho,
    }


pipeline_artifacts = load_pipeline_models(PIPELINE_MODELS_DIR)

In [ ]:
SIGNAL_COLS = ["ACXXX_SN1 [g]", "ACXXX_SN2 [g]"]
TACHO_COL   = "Tacho_Ch8 [Hz]"


@torch.no_grad()
def predict_wybieg(file_info, artifacts, fs=32768, window_s=WINDOW_S):
    """
    Przepuszcza jeden plik wybiegowy przez pipeline i zwraca predykcje obu modeli.

    Kroki:
      1. Wczytanie CSV i przycięcie do okna quasi-ustalonego
      2. Obliczenie spektrogramu zespolonego (STFT) dla kanałów SN1, SN2 [+ Tacho]
      3. Normalizacja amplitudy zgodna ze statystykami pipeline
      4. Inferencja modelu non_tacho → wartość niewyważenia
      5. Inferencja modelu tacho (jeśli sygnał dostępny) → kąt [sin, cos]
    """
    df = pd.read_csv(file_info["path"])

    non_tacho_model = artifacts["non_tacho_model"]
    tacho_model     = artifacts["tacho_model"]
    norm_nt         = artifacts["norm_non_tacho"]
    norm_ta         = artifacts["norm_tacho"]

    # Non-tacho: SN1 + SN2 (6 kanałów)
    X_nt, names_nt, freq_hz = build_spectrogram_tensor(
        df, fs, channels=SIGNAL_COLS, window_s=window_s
    )
    X_nt = apply_pipeline_normalization(X_nt, names_nt, norm_nt)

    # Dodanie wymiaru batch: [1, C, F, T]
    tensor_nt = torch.from_numpy(X_nt).unsqueeze(0).to(DEVICE)
    pred_imbalance = float(non_tacho_model(tensor_nt).cpu().squeeze())

    result = {
        "title":           file_info["title"],
        "true_imbalance":  file_info["true_imbalance"],
        "pred_imbalance":  pred_imbalance,
        # Zaokrąglenie do klasy poprzez round() – model jest regresyjny
        "pred_class":      int(round(np.clip(pred_imbalance, 0, 3))),
        "pred_angle_deg":  None,
        "true_angle":      file_info["angle"],
    }

    # Tacho: SN1 + SN2 + Tacho (9 kanałów) 
    if file_info["has_tacho"]:
        X_ta, names_ta, _ = build_spectrogram_tensor(
            df, fs, channels=SIGNAL_COLS + [TACHO_COL], window_s=window_s
        )
        X_ta = apply_pipeline_normalization(X_ta, names_ta, norm_ta)

        tensor_ta = torch.from_numpy(X_ta).unsqueeze(0).to(DEVICE)
        sin_cos = tacho_model(tensor_ta).cpu().squeeze().numpy()

        # Konwersja sin/cos → kąt w stopniach (0–360°)
        pred_angle_rad = float(np.arctan2(sin_cos[0], sin_cos[1]))
        pred_angle_deg = float(np.degrees(pred_angle_rad) % 360)
        result["pred_angle_deg"] = pred_angle_deg

    return result, freq_hz, X_nt, names_nt

In [ ]:
IMBALANCE_LABELS = {0: "wyważone (0)", 1: "zew (1)", 2: "zew2 (2)", 3: "zew+sr (3)"}

results_table = []

for file_info in WYBIEG_FILES:
    print(f"\nPrzetwarzanie: {file_info['title']} ...")

    result, freq_hz, X_nt, names_nt = predict_wybieg(
        file_info, pipeline_artifacts, window_s=WINDOW_S
    )
    results_table.append(result)

    # Wydruk szczegółowy dla bieżącego pliku
    correct = "✓" if result["pred_class"] == result["true_imbalance"] else "✗"
    print(f"  Niewyważenie  | prawda: {IMBALANCE_LABELS[result['true_imbalance']]:15s} "
          f"| predykcja raw: {result['pred_imbalance']:+.3f} "
          f"| klasa: {result['pred_class']} {correct}")

    if result["pred_angle_deg"] is not None:
        # Kąt prawdziwy zakodowany w systemie 17-podziałów (zgodnie z pipeline)
        true_angle_rad = 2 * np.pi * result["true_angle"] / 17
        true_angle_deg = np.degrees(true_angle_rad) % 360
        angle_error    = abs(np.degrees(np.arctan2(
            np.sin(np.radians(result["pred_angle_deg"]) - true_angle_rad),
            np.cos(np.radians(result["pred_angle_deg"]) - true_angle_rad)
        )))
        print(f"  Kąt (tacho)   | prawda: {true_angle_deg:.1f}°  "
              f"| predykcja: {result['pred_angle_deg']:.1f}°  "
              f"| błąd kołowy: {angle_error:.1f}°")

    # Podgląd spektrogramu amplitude_db dla SN2 po normalizacji
    amp_idx = names_nt.index("ACXXX_SN2 [g] | amplitude_db")
    plt.figure(figsize=(10, 3))
    plt.pcolormesh(
        np.arange(X_nt.shape[-1]),
        freq_hz,
        X_nt[amp_idx],
        shading="gouraud",
        cmap="viridis",
    )
    plt.yscale("log")
    plt.ylim(PIPELINE_F_MIN_HZ, PIPELINE_F_MAX_HZ)
    plt.colorbar(label="Amplituda [dB, znorm.]")
    plt.title(f"SN2 amplitude_db (po normalizacji) – {file_info['title']}, "
              f"okno {WINDOW_S}s | pred_imbalance={result['pred_imbalance']:.3f}")
    plt.xlabel("Bin czasowy STFT")
    plt.ylabel("Częstotliwość [Hz]")
    plt.tight_layout()
    plt.show()

# Zbiorcze podsumowanie w formie tabeli
print("\n" + "=" * 70)
print("PODSUMOWANIE INFERENCJI NA PLIKACH WYBIEGOWYCH")
print("=" * 70)
df_results = pd.DataFrame(results_table)
df_results["correct"] = df_results["pred_class"] == df_results["true_imbalance"]
display(df_results[["title", "true_imbalance", "pred_imbalance", "pred_class",
                     "correct", "true_angle", "pred_angle_deg"]])

---
## Wizualizacja pełnych komponentów spektrogramu dla wybiegów

Analogicznie do plot_first_file_component` z **Przygotowanie_Danych_V3**, poniżej przedstawiamy wszystkie trzy komponenty spektralne (amplitude_db, phase_sin, phase_cos) dla każdego kanału (SN1, SN2 oraz Tacho jeśli dostępny) dla każdego pliku wybiegowego.

Użyto tych samych parametrów STFT co w pipeline (nperseg=2^15), pełnego nagrania (window_s=None) oraz logarytmicznej skali osi częstotliwości (1–5000 Hz).

In [ ]:
def plot_wybieg_components(file_info, fs=32768):
    """
    Wizualizuje wszystkie trzy komponenty spektralne (amplitude_db, phase_sin, phase_cos)
    dla każdego kanału (SN1, SN2 i opcjonalnie Tacho) — analogicznie do
    plot_first_file_components z Przygotowanie_Danych_V3.

    Układ siatki: wiersze = {amplitude_db, phase_sin, phase_cos}, kolumny = kanały.
    Pełne nagranie (window_s=None), skala logarytmiczna osi częstotliwości.
    """
    df = pd.read_csv(file_info["path"])

    channels = ["ACXXX_SN1 [g]", "ACXXX_SN2 [g]"]
    if file_info["has_tacho"] and "Tacho_Ch8 [Hz]" in df.columns:
        channels.append("Tacho_Ch8 [Hz]")

    X, component_names, frequency_hz = build_spectrogram_tensor(
        df, fs, channels=channels, window_s=None
    )

    component_types = ["amplitude_db", "phase_sin", "phase_cos"]
    n_channels = len(channels)
    n_rows = len(component_types)

    fig, axes = plt.subplots(
        n_rows, n_channels,
        figsize=(5 * n_channels, 4 * n_rows),
        squeeze=False,
    )
    fig.suptitle(
        f"Komponenty spektrogramu — {file_info['title']} (pełne nagranie)",
        fontsize=14, fontweight="bold", y=1.01,
    )

    # Parametry kolormap i zakresów per komponent
    cmap_cfg = {
        "amplitude_db": {"cmap": "viridis", "vmin": None, "vmax": None, "label": "Amplituda [dB]"},
        "phase_sin":    {"cmap": "viridis",  "vmin": -1,   "vmax": 1,   "label": "sin(faza)"},
        "phase_cos":    {"cmap": "viridis",  "vmin": -1,   "vmax": 1,   "label": "cos(faza)"},
    }

    t_bins = np.arange(X.shape[-1])

    for row_idx, comp_type in enumerate(component_types):
        for col_idx, ch in enumerate(channels):
            comp_name = f"{ch} | {comp_type}"
            chan_idx = component_names.index(comp_name)
            data = X[chan_idx]  # [F, T]

            cfg = cmap_cfg[comp_type]
            ax = axes[row_idx][col_idx]

            im = ax.pcolormesh(
                t_bins,
                frequency_hz,
                data,
                shading="gouraud",
                cmap=cfg["cmap"],
                vmin=cfg["vmin"],
                vmax=cfg["vmax"],
            )
            ax.set_yscale("log")
            ax.set_ylim(PIPELINE_F_MIN_HZ, PIPELINE_F_MAX_HZ)

            # Tytuł tylko w pierwszym wierszu, etykieta komponentu tylko w pierwszej kolumnie
            if row_idx == 0:
                ax.set_title(ch, fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(f"{comp_type}\nCzęst. [Hz]", fontsize=9)
            else:
                ax.set_ylabel("")

            ax.set_xlabel("Bin czasowy STFT" if row_idx == n_rows - 1 else "")
            plt.colorbar(im, ax=ax, label=cfg["label"], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


# Wizualizacja dla wszystkich plików wybiegowych
for file_info in WYBIEG_FILES:
    print(f"Generowanie wykresów: {file_info['title']} ...")
    plot_wybieg_components(file_info)

In [ ]:
# Przygotowanie tabel predykcji (format zgodny z Modele_V3)

ANGLE_DIVISIONS = 17

imbalance_rows = []
angle_rows = []

for r in results_table:
    imbalance_rows.append({
        "title":      r["title"],
        "true_value": float(r["true_imbalance"]),
        "pred_value": float(r["pred_imbalance"]),
        "abs_error":  abs(float(r["pred_imbalance"]) - float(r["true_imbalance"])),
    })

    if r["pred_angle_deg"] is not None:
        true_angle_rad = 2 * np.pi * r["true_angle"] / ANGLE_DIVISIONS
        true_angle_deg = float(np.degrees(true_angle_rad) % 360)
        pred_angle_deg = float(r["pred_angle_deg"])

        diff = np.arctan2(
            np.sin(np.radians(pred_angle_deg) - true_angle_rad),
            np.cos(np.radians(pred_angle_deg) - true_angle_rad),
        )

        angle_rows.append({
            "title":           r["title"],
            "true_angle_deg":  true_angle_deg,
            "pred_angle_deg":  pred_angle_deg,
            "angle_error_deg": float(abs(np.degrees(diff))),
            "true_sin":        float(np.sin(true_angle_rad)),
            "true_cos":        float(np.cos(true_angle_rad)),
            "pred_sin":        float(np.sin(np.radians(pred_angle_deg))),
            "pred_cos":        float(np.cos(np.radians(pred_angle_deg))),
        })

wybieg_imbalance_table = pd.DataFrame(imbalance_rows)
wybieg_angle_table     = pd.DataFrame(angle_rows)

#print("Tabela niewyważenia:")
#display(wybieg_imbalance_table)

#if len(wybieg_angle_table):
#    print("Tabela kąta (tylko próbki z tachometrem):")
#    display(wybieg_angle_table)


# plot_imbalance_regression_results (adaptacja dla wybiegów)

def plot_imbalance_regression_results_wybieg(pred_table, title):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.scatter(pred_table["true_value"], pred_table["pred_value"], alpha=0.9, zorder=5)
    #for _, row in pred_table.iterrows():
        #ax.annotate(row["title"], (row["true_value"], row["pred_value"]),
                    #textcoords="offset points", xytext=(5, 5), fontsize=8)
    min_val = min(pred_table["true_value"].min(), pred_table["pred_value"].min()) - 0.3
    max_val = max(pred_table["true_value"].max(), pred_table["pred_value"].max()) + 0.3
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="gray")
    ax.set_xlabel("True imbalance / mass value")
    ax.set_ylabel("Predicted imbalance / mass value")
    ax.set_title("Predicted vs true value")

    ax = axes[0, 1]
    ax.bar(pred_table["title"], pred_table["abs_error"])
    ax.set_xlabel("Próbka")
    ax.set_ylabel("Absolute error")
    ax.set_title("Błąd bezwzględny per próbka")
    ax.tick_params(axis="x", rotation=30)

    ax = axes[1, 0]
    order = np.argsort(pred_table["true_value"].to_numpy())
    x = np.arange(len(order))
    ax.plot(x, pred_table["true_value"].to_numpy()[order], marker="o", label="true")
    ax.plot(x, pred_table["pred_value"].to_numpy()[order], marker="s", label="pred")
    ax.set_xticks(x)
    ax.set_xticklabels(pred_table["title"].to_numpy()[order], rotation=30, fontsize=8)
    ax.set_ylabel("Imbalance / mass value")
    ax.set_title("True and predicted values")
    ax.legend()

    ax = axes[1, 1]
    residual = pred_table["pred_value"] - pred_table["true_value"]
    ax.scatter(pred_table["true_value"], residual, alpha=0.9, zorder=5)
    # for _, row in pred_table.iterrows():
    #     ax.annotate(row["title"], (row["true_value"], row["pred_value"] - row["true_value"]),
    #                 textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.axhline(0, linestyle="--", color="gray")
    ax.set_xlabel("True imbalance / mass value")
    ax.set_ylabel("Residual: pred - true")
    ax.set_title("Residuals")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_imbalance_regression_results_wybieg(
    wybieg_imbalance_table,
    "Wybiegi → non-tacho model: predykcja niewyważenia",
)


# plot_angle_regression_results (adaptacja dla wybiegów)

def plot_angle_regression_results_wybieg(pred_table, title):
    if len(pred_table) == 0:
        print("Brak próbek z tachometrem — wykres kąta pominięty.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.scatter(pred_table["true_angle_deg"], pred_table["pred_angle_deg"], alpha=0.9, zorder=5)
    # for _, row in pred_table.iterrows():
    #     ax.annotate(row["title"], (row["true_angle_deg"], row["pred_angle_deg"]),
    #                 textcoords="offset points", xytext=(5, 5), fontsize=9)
    ax.plot([0, 360], [0, 360], linestyle="--", color="gray")
    ax.set_xlabel("True angle [deg]")
    ax.set_ylabel("Predicted angle [deg]")
    ax.set_title("Predicted vs true angle")
    ax.set_xlim(0, 360)
    ax.set_ylim(0, 360)

    ax = axes[0, 1]
    ax.bar(pred_table["title"], pred_table["angle_error_deg"])
    ax.set_xlabel("Próbka")
    ax.set_ylabel("Circular error [deg]")
    ax.set_title("Błąd kołowy kąta per próbka")
    ax.tick_params(axis="x", rotation=30)

    ax = axes[1, 0]
    ax.scatter(pred_table["true_sin"], pred_table["pred_sin"], alpha=0.9, label="sin", zorder=5)
    ax.scatter(pred_table["true_cos"], pred_table["pred_cos"], alpha=0.9, label="cos", zorder=5)
    # for _, row in pred_table.iterrows():
    #     ax.annotate(row["title"], (row["true_sin"], row["pred_sin"]),
    #                 textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.plot([-1, 1], [-1, 1], linestyle="--", color="gray")
    ax.set_xlabel("True value")
    ax.set_ylabel("Predicted value")
    ax.set_title("Predicted vs true sin/cos")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.legend()

    ax = axes[1, 1]
    ax.remove()
    ax_polar = fig.add_subplot(2, 2, 4, projection="polar")
    for _, row in pred_table.iterrows():
        true_rad = np.radians(row["true_angle_deg"])
        pred_rad = np.radians(row["pred_angle_deg"])
        ax_polar.annotate("", xy=(pred_rad, 1.0), xytext=(0, 0),
                          arrowprops=dict(arrowstyle="->", color="tab:orange", lw=2))
        ax_polar.annotate("", xy=(true_rad, 0.8), xytext=(0, 0),
                          arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2))
        ax_polar.text(pred_rad, 1.1, row["title"], fontsize=8, ha="center")
    ax_polar.set_title("Kąt: true (niebieski) vs pred (pomarańczowy)", pad=15)
    ax_polar.set_ylim(0, 1.2)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_angle_regression_results_wybieg(
    wybieg_angle_table,
    "Wybiegi → tacho model: predykcja kąta niewyważenia",
)
